In [ ]:
import sys
import boto3
from pathlib import Path
from google.transit import gtfs_realtime_pb2
import requests

BUCKET = "busybusybusy-dc"
PREFIX = "wmata/vehicle_positions"
REGION = "us-east-1"

In [ ]:
def fetch_files_by_date(date):
    '''
    YYYY-MM-DD format
    '''
    s3 = boto3.client("s3", region_name=REGION)

    date_prefix = f"{PREFIX}/{date}"
    paginator = s3.get_paginator("list_objects_v2")
    keys = [
        obj["Key"]
        for page in paginator.paginate(Bucket=BUCKET, Prefix=date_prefix)
        for obj in page.get("Contents", [])
        if obj["Key"].endswith(".pb")
    ]

    if not keys:
        print(f"no objects found for date {date}")
        return

    output_dir = Path(f"./pb_downloads/{date}")
    output_dir.mkdir(parents=True, exist_ok=True)

    for key in keys:
        destination = output_dir / key.split("/")[-1]
        s3.download_file(BUCKET, key, str(destination))
        print(f"downloaded {key} -> {destination}")

In [ ]:

def decode_protobuf_file(pb_filepath):
